# Cell 1: import, scan, select, laterality

In [ ]:
import sys, glob, time, json, datetime
from pathlib import Path
import pandas as pd

hits = glob.glob("/kaggle/usr/lib/**/knee_prep.py", recursive=True)
print(hits)
sys.path.insert(0, str(Path(hits[0]).parent))
import knee_prep as kp

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
OUT = Path("/kaggle/working")
LIMIT = None

ser = pd.read_csv(f"{ROOT}/train_series.csv")
t0 = time.time()
inv = kp.scan_all(ser, ROOT, "train_series")
print(f"scanned {len(inv)} series in {time.time() - t0:.0f}s, errors: {inv['error'].notna().sum()}")

sel = kp.select_series(inv)
lat = kp.study_laterality(inv)
print("selected:", len(sel), " studies:", sel["StudyInstanceUID"].nunique())
print(lat["side_src"].value_counts().to_string())

inv.to_csv(OUT / "series_scan.csv", index=False)
sel.to_csv(OUT / "selected_series.csv", index=False)
lat.to_csv(OUT / "laterality.csv", index=False)

if LIMIT:
    sel = sel[sel["StudyInstanceUID"].isin(sel["StudyInstanceUID"].unique()[:LIMIT])]

# Cell 2: build

In [ ]:
t0 = time.time()
qa = kp.build_cache(sel, ROOT, "train_series", OUT / "cache", workers=4)
print(f"built in {(time.time() - t0) / 60:.1f} min")
qa.to_csv(OUT / "cache_qa.csv", index=False)
print("npz files:", len(list((OUT / "cache").glob("*.npz"))))
print("errors:", qa["error"].notna().sum() if len(qa) else 0)
if len(qa):
    print(qa[["dup_pos", "big_gaps", "mixed_shapes"]].sum().to_string())

# Cell 3: lineage record

In [ ]:
meta = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "source": "rsna-knee-abnormality-detection/train_series",
    "git_commit": "b1f7e63",
    "shape": [kp.D, kp.S, kp.S], "dtype": "uint8", "keys": ["sag", "cor", "axi"],
    "selection": "readable first, FS first, thickness closest to 3 mm, then most rows",
    "intensity": "per-volume 0.5-99.5 percentile clip, scaled to 0-255",
    "geometry": "pad to square, antialiased resize, linear resample along depth",
    "orientation": "native, identical per plane across dataset; flip left knees at load time using laterality.csv",
    "known_issues": ["knee scale varies with field of view", "1098 studies with unknown side"],
}
json.dump(meta, open(OUT / "cache_meta.json", "w"), indent=2)